# Fase 5 — Avaliação (CRISP-DM)

O guia define o objetivo desta fase: medir os resultados **pela lente dos critérios de sucesso definidos na Fase 1** (impacto real > precisão técnica). O notebook faz três verificações:

1. **Fidelidade dos dados:** a base extraída reproduz os números publicados pela SSP-DF?
2. **Robustez da resposta:** o ranking muda se mudarmos o recorte (só 2025, soma 2024+2025, 1º semestre)?
3. **Critérios de sucesso da Fase 1:** cada um foi atendido?

In [1]:
from pathlib import Path
import pandas as pd
from scipy.stats import spearmanr

RAIZ = Path('..').resolve()
PROC = RAIZ / 'dados' / 'processados'
ANO_BASE, ANO_ALVO = 2024, 2025
base = pd.read_csv(PROC / 'violencia_domestica_ra.csv')
log = pd.read_csv(PROC / 'log_validacao.csv')
res = pd.read_csv(PROC / f'resultado_modelagem_{ANO_BASE}_{ANO_ALVO}.csv', index_col='ra')
anual = base[base['periodo'] == 'Anual'].pivot(index='ra', columns='ano', values='registros')
sem = base[base['periodo'] == '1º semestre'].pivot(index='ra', columns='ano', values='registros')
verificacoes = []
def verificar(nome, ok, detalhe):
    verificacoes.append({'verificação': nome, 'resultado': 'OK' if ok else 'FALHOU', 'detalhe': detalhe})
    print(('✔ ' if ok else '✘ ') + nome + ' — ' + detalhe)

## 5.1 Fidelidade dos dados extraídos

In [2]:
erros = log[log['tipo'] == 'ERRO']
verificar('Somas por RA = TOTAL de cada relatório', erros.empty,
          f"{(log['ra']=='TOTAL').sum()} totais conferidos, {len(erros)} divergências")

# Valores registrados na Fase 2 a partir da leitura do RAC-001/2026 (p. 1)
fase2_totais = {2024: 21721, 2025: 23066}
for ano, esperado in fase2_totais.items():
    obtido = int(anual[ano].sum())
    verificar(f'Total DF {ano} igual ao da Fase 2', obtido == esperado, f'extraído {obtido} | Fase 2 {esperado}')

fase2_top10 = {'Ceilândia': 3135, 'Planaltina': 1951, 'Samambaia': 1809, 'Taguatinga': 1339,
               'Recanto das Emas': 1303, 'Gama': 1238, 'Sol Nascente/Pôr do Sol': 1233,
               'Santa Maria': 1198, 'São Sebastião': 1086, 'Brasília': 959}
dif = {ra: int(anual.loc[ra, 2025]) - v for ra, v in fase2_top10.items() if int(anual.loc[ra, 2025]) != v}
verificar('Top 10 de 2025 igual ao da Fase 2', not dif, 'sem diferenças' if not dif else str(dif))

top10_pct = 100 * anual[ANO_ALVO].nlargest(10).sum() / anual[ANO_ALVO].sum()
verificar('Concentração das 10 maiores ≈ 66% (citado no relatório)', round(top10_pct) == 66, f'{top10_pct:.1f}%')

✔ Somas por RA = TOTAL de cada relatório — 10 totais conferidos, 0 divergências
✔ Total DF 2024 igual ao da Fase 2 — extraído 21721 | Fase 2 21721
✔ Total DF 2025 igual ao da Fase 2 — extraído 23066 | Fase 2 23066
✔ Top 10 de 2025 igual ao da Fase 2 — sem diferenças
✔ Concentração das 10 maiores ≈ 66% (citado no relatório) — 66.1%


### Consistência entre relatórios diferentes

O 1º semestre de 2025 aparece em outro relatório (RAC-020/2026). Se a extração estiver correta, o semestre deve representar **cerca de metade** do ano em cada RA. Grandes desvios indicariam erro de leitura ou revisão forte dos dados.

In [3]:
if ANO_ALVO in sem.columns:
    razao = (sem[ANO_ALVO] / anual[ANO_ALVO]).rename('1º sem / ano')
    df_razao = sem[ANO_ALVO].sum() / anual[ANO_ALVO].sum()
    fora = razao[(razao < 0.40) | (razao > 0.60)]
    verificar('1º semestre ≈ metade do ano em cada RA', len(fora) <= 3,
              f'DF: {df_razao:.1%}; RAs fora de 40–60%: {", ".join(fora.index) or "nenhuma"}')
    display(razao.sort_values().to_frame().T.round(2))

✔ 1º semestre ≈ metade do ano em cada RA — DF: 50.8%; RAs fora de 40–60%: Sudoeste


ra,Sudoeste,SIA,Park Way,Arniqueira,Riacho Fundo II,Estrutural,Guará,Lago Norte,Núcleo Bandeirante,Fercal,...,Sol Nascente/Pôr do Sol,Águas Claras,Taguatinga,Itapoã,Varjão,Gama,Cruzeiro,Brazlândia,Lago Sul,Jardim Botânico
1º sem / ano,0.39,0.41,0.44,0.44,0.47,0.47,0.47,0.48,0.48,0.48,...,0.52,0.52,0.52,0.53,0.53,0.54,0.56,0.57,0.59,0.6


## 5.2 Robustez da resposta

In [4]:
top_alvo = set(res.nsmallest(10, f'pos_{ANO_ALVO}').index)
top_base = set(res.nsmallest(10, f'pos_{ANO_BASE}').index)
top_soma = set(res.nsmallest(10, 'pos_2_anos').index)
verificar(f'Mesmas 10 RAs em {ANO_BASE} e {ANO_ALVO}', top_alvo == top_base,
          'iguais' if top_alvo == top_base else f'entram {top_alvo-top_base}, saem {top_base-top_alvo}')
verificar('Top 10 de 2025 = top 10 da soma 2024+2025', top_alvo == top_soma,
          'iguais' if top_alvo == top_soma else f'diferença: {top_alvo ^ top_soma}')
rho = spearmanr(res[f'reg_{ANO_BASE}'], res[f'reg_{ANO_ALVO}']).statistic
verificar(f'Ranking {ANO_BASE} × {ANO_ALVO} estável (Spearman ≥ 0,9)', rho >= 0.9, f'ρ = {rho:.3f}')
lider = res[f'pos_{ANO_ALVO}'].idxmin()
anos_lider = [a for a in anual.columns if anual[a].idxmax() == lider]
verificar(f'{lider} lidera em todos os anos da série', len(anos_lider) == anual.shape[1],
          f'lidera em {anos_lider}')

✔ Mesmas 10 RAs em 2024 e 2025 — iguais
✔ Top 10 de 2025 = top 10 da soma 2024+2025 — iguais
✔ Ranking 2024 × 2025 estável (Spearman ≥ 0,9) — ρ = 0.992
✔ Ceilândia lidera em todos os anos da série — lidera em [2018, 2019, 2020, 2021, 2022, 2024, 2025]


## 5.3 Critérios de sucesso da Fase 1

In [5]:
criterios = pd.DataFrame([
    ('1. Identificar as RAs com maiores números absolutos de registros', 'Atendido',
     'Ranking completo das 33 RAs (Fase 4, item 4.1; fig1_top10_ras.png)'),
    ('2. Comparar 2024 e 2025 de forma verificável', 'Atendido',
     'Variação por RA (4.3) com base extraída e validada contra os totais oficiais (5.1)'),
    ('3. Gráficos e tabelas claros, com fonte e período', 'Atendido',
     'Figuras 1–5 com fonte SSP-DF no rodapé; planilha com dicionário de dados'),
    ('4. Explicar limitações e não tratar números absolutos como risco', 'Atendido',
     'Seção de limitações em Avaliação.md; taxas por habitante ficam como próxima iteração'),
], columns=['critério', 'situação', 'evidência'])
display(criterios)
pd.DataFrame(verificacoes)

,critério,situação,evidência
0,1. Identificar as RAs com maiores números abso...,Atendido,"Ranking completo das 33 RAs (Fase 4, item 4.1;..."
1,2. Comparar 2024 e 2025 de forma verificável,Atendido,Variação por RA (4.3) com base extraída e vali...
2,"3. Gráficos e tabelas claros, com fonte e período",Atendido,Figuras 1–5 com fonte SSP-DF no rodapé; planil...
3,4. Explicar limitações e não tratar números ab...,Atendido,Seção de limitações em Avaliação.md; taxas por...


,verificação,resultado,detalhe
0,Somas por RA = TOTAL de cada relatório,OK,"10 totais conferidos, 0 divergências"
1,Total DF 2024 igual ao da Fase 2,OK,extraído 21721 | Fase 2 21721
2,Total DF 2025 igual ao da Fase 2,OK,extraído 23066 | Fase 2 23066
3,Top 10 de 2025 igual ao da Fase 2,OK,sem diferenças
4,Concentração das 10 maiores ≈ 66% (citado no r...,OK,66.1%
5,1º semestre ≈ metade do ano em cada RA,OK,DF: 50.8%; RAs fora de 40–60%: Sudoeste
6,Mesmas 10 RAs em 2024 e 2025,OK,iguais
7,Top 10 de 2025 = top 10 da soma 2024+2025,OK,iguais
8,"Ranking 2024 × 2025 estável (Spearman ≥ 0,9)",OK,ρ = 0.992
9,Ceilândia lidera em todos os anos da série,OK,"lidera em [2018, 2019, 2020, 2021, 2022, 2024,..."


In [6]:
assert all(v['resultado'] == 'OK' for v in verificacoes if 'Fase 2' in v['verificação'] or 'TOTAL' in v['verificação']), \
    'Falha de fidelidade: revisar a extração antes de publicar'
pd.DataFrame(verificacoes).to_csv(PROC / 'avaliacao_verificacoes.csv', index=False, encoding='utf-8-sig')